In [ ]:
import sys
import os
import json
import glob
from pathlib import Path

def find_project_root(marker_files={'Data', 'src', '.git'}):
    """
    Locates the project root by searching upwards for specific markers.
    """
    current_path = Path.cwd().resolve()
    
    # Search upwards to the filesystem root
    for parent in [current_path] + list(current_path.parents):
        # Check if any of the marker files/folders exist in this directory
        if any((parent / marker).exists() for marker in marker_files):
            return parent
            
    return None

# Execute the search
root = find_project_root()

if not root:
    # Instead of a hardcoded path, consider an environment variable 
    # for local dev environments where the structure might be unusual.
    fallback = os.getenv("PROJECT_ROOT_FALLBACK")
    if fallback and Path(fallback).exists():
        root = Path(fallback)

if root:
    project_root = str(root)
    os.chdir(project_root)
    sys.path.insert(0, project_root)
    print(f"Project root set to: {project_root}")
else:
    raise RuntimeError("Could not determine project root. Ensure you are running from within the project directory.")

from src.id_converter import numeric_to_url_id
from src.retrieve_event_data import retrieve_event_data
import time

# Debug: Check current directory and list files
print(f"Current working directory: {os.getcwd()}")
print(f"Files in ./Data/raw/events/: {os.listdir('./Data/raw/events/')[:10]}")

# Pull event IDs from JSON files in Data/raw/events
event_files = glob.glob("./Data/raw/events/event_*.json")
print(f"Glob found {len(event_files)} files")
target_ids = []

for event_file in sorted(event_files):
    with open(event_file, 'r') as f:
        data = json.load(f)
        # Extract the event ID from the JSON data (it's "EventId", not "id")
        if 'EventId' in data:
            target_ids.append(data['EventId'])

print(f"Found {len(target_ids)} events: {target_ids[:10]}...")  # Show first 10

for num_id in target_ids:
    url_id = numeric_to_url_id(num_id)
    print(f"Starting job for ID: {num_id}")
    
    # The function handles everything and creates the file
    retrieve_event_data(url_id)
    time.sleep(5)